In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_groq import ChatGroq


In [8]:
llmGroq=ChatGroq(
  model="llama-3.3-70b-versatile",
  api_key=os.getenv("GROQ_API_KEY"),
  temperature=0.2,
)


In [ ]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_history.db"
    )

chain_with_memory = RunnableWithMessageHistory(
    llmGroq,
    get_session_history,
)

### Mini Project - Smart Query Router

In [4]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnablePassthrough
load_dotenv()


True

In [6]:
llmGroq=ChatGroq(
  model="llama-3.3-70b-versatile",
  api_key=os.getenv("GROQ_API_KEY"),
  temperature=0.2,
)


In [25]:
# 1. Math chain - locked to concise numeric answers
math_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math tutor. Give ONLY the final numeric answer and a one-line explanation. No fluff."),
    ("user", "{question}")
])
math_chain = math_prompt | llmGroq.bind(temperature=0) | StrOutputParser()
# .bind(temperature=0) forces deterministic math answers regardless of the LLM's default temperature

# 2. Code chain - locked to code-focused responses
code_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior software engineer. Answer with a code example first, then a brief explanation."),
    ("user", "{question}")
])
code_chain = code_prompt | llmGroq | StrOutputParser()

# 3. General chain - plain conversational assistant
general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly, helpful assistant. Answer clearly and concisely."),
    ("user", "{question}")
])
def log_general_route(input_dict: dict) -> dict:
    """Logs fallback routing before passing payload forward."""
    print("[ROUTER LOG]: 💬 Routed to GENERAL CHAIN (Default Fallback)",flush=True)
    return input_dict

general_chain = (
    RunnableLambda(log_general_route) 
    | general_prompt 
    | llmGroq 
    | StrOutputParser()
)

In [26]:
def is_math_question(input_dict):
    question = input_dict["question"].lower()
    keywords = ["calculate", "solve", "how much is", "+", "-", "*", "/", "sum", "average"]
    return any(k in question for k in keywords)

def is_code_question(input_dict):
    question = input_dict["question"].lower()
    keywords = ["code", "python", "function", "bug", "error", "script", "debug"]
    return any(k in question for k in keywords)

router_chain = RunnableBranch(
    (is_math_question, math_chain),
    (is_code_question, code_chain),
    general_chain  # default fallback if neither condition matches
)

In [28]:
questions = [
    "   Calculate 245 * 12   ",
    "Write a Python function to reverse a string",
    "What's the capital of France?",
    "Hi, How are you doing today?"
]

for q in questions:
    print(f"\n--- Question: {q} ---")
    result = router_chain.invoke({"question": q})
    print(result)


--- Question:    Calculate 245 * 12    ---
2940, result of multiplying 245 by 12.

--- Question: Write a Python function to reverse a string ---
```python
def reverse_string(input_str: str) -> str:
    """
    Reverses the input string.

    Args:
        input_str (str): The string to be reversed.

    Returns:
        str: The reversed string.
    """
    return input_str[::-1]

# Example usage:
print(reverse_string("Hello World"))  # Output: "dlroW olleH"
```

This function uses Python's slice notation to extract the characters of the input string in reverse order. The `[::-1]` slice means "start at the end of the string and end at position 0, move with the step -1" which effectively reverses the string.

--- Question: What's the capital of France? ---
[ROUTER LOG]: 💬 Routed to GENERAL CHAIN (Default Fallback)
The capital of France is Paris.

--- Question: Hi, How are you doing today? ---
[ROUTER LOG]: 💬 Routed to GENERAL CHAIN (Default Fallback)
I'm doing well, thanks for asking. 

### Chain with just RunnablePassthrough() will output the original input without any modification.

In [30]:
from langchain_core.runnables import RunnablePassthrough
chain = RunnablePassthrough()

In [31]:
chain.invoke("Hello")

'Hello'

### RunnableLambda

 -To use a custom function inside a LCEL chain we need to wrap it up with RunnableLambda.


In [32]:
def greeting(name: str) -> str:
    return f"{name} is a great name!"

In [33]:
from langchain_core.runnables import RunnableLambda

chain=RunnablePassthrough() | RunnableLambda(greeting)

In [34]:
chain.invoke("Harsh")

'Harsh is a great name!'

## RunnableParallel

We will use `RunnableParallel` for running tasks in parallel. This is one of the most important and useful `Runnable` classes in LangChain.

In the following chain, `RunnableParallel` executes two tasks concurrently:
* **`operation_a`**: Uses `RunnablePassthrough` to forward the raw input.
* **`operation_b`**: Uses `RunnableLambda` wrapped around the `russian_lastname` function.

In [36]:
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "operation_b": RunnableLambda(greeting)
    }
)

In [37]:
chain.invoke("Harsh")

{'operation_a': 'Harsh', 'operation_b': 'Harsh is a great name!'}

In [39]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq  
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# Free, local embeddings — no API key needed
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_texts(
    ["MLCourses focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."],
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llmGroq
    | StrOutputParser()
)

response = retrieval_chain.invoke("What is MLCourses?")
print(response)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3087.71it/s]


MLCourses focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English.
